# Práctica: Fundamentos de LLMs y Configuración de la API

En esta sesión dejaremos atrás los modelos tradicionales y experimentaremos con los Large Language Models (LLMs) directamente a través del código. Utilizaremos el modelo **Gemini de Google**, ya que su nivel gratuito ofrece una amplia capacidad.

Contenidos:
1. Crear Gemini API Key.
2. Uso de parámetros de Gemini.
3. Creación de buenos prompts para evitar alucinaciones.
4. Uso del modo chat de Gemini para guardar un historial.
5. Usar Gemini como Agente de IA.

## 1. Obtener la API Key de Gemini

Para que nuestro código en Python pueda comunicarse con el modelo de Gemini que está alojado en los servidores de Google, necesitamos unas "credenciales" únicas: la **API Key**.

### Pasos a seguir:
1. Ve a la web de **Google AI Studio**: [https://aistudio.google.com/](https://aistudio.google.com/) e inicia sesión con una cuenta de Google (preferiblemente con la que vayas a ejecutar este Colab).
2. En el menú de la izquierda, haz clic en el botón **"Get API key"**.

![Obtener API Key](https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/3_llm/img/gemini_api_key.png)

3. Haz clic en **"Crear Clave de API"**.

![Crear API Key](https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/3_llm/img/crear_api_key.png)

4. Si es la primera vez que hacemos una, es necesario crear un proyecto donde relacionar la API Key.

![Crear Proyecto](https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/3_llm/img/crear_proyecto.png)

5. Hay que asignarle un nombre y pulsar **"Crear Proyecto"** y **"Crear Clave"**.

![Asignar nombre](https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/3_llm/img/proyecto_nombre.png)

6. Se generará una cadena de texto larga (algo parecido a `AIzaSyB...`). **Cópiala y guárdala**. ¡No la compartas públicamente!

![API Key](https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/3_llm/img/api_key.png)

---

## 2. Instalación de dependencias
Vamos a instalar el SDK oficial de Google para Python, que nos facilitará enormemente interactuar con su API.

In [ ]:
!pip install -q -U google-genai

---
## 3. Configuración Inicial
Importamos la librería y NO usaremos nuestra API Key directamente en el código. Iremos a los secrets de Google Colab.
![Colab Secret](https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/3_llm/img/colab_secret.png)

Otra opción es añadirla manualmente en "Añadir secreto nuevo" con el nombre `GEMINI_API_KEY`

**Importante**: Cuidado con usar cuentas institucionales (@educarex... etc), puede que no tengan permisos para crear API Keys.

In [ ]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=GOOGLE_API_KEY)

#Instanciamos el modelo. Usaremos gemini-3.1-flash-lite-preview, hay varias versiones gratuitas, cada una con su rate limit. Usaremos esta al tener 500 peticiones diarias
MODEL_ID = "models/gemini-3.1-flash-lite-preview"

for m in client.models.list():
    print(f"Modelo disponible: {m.name}")

---
## 4. Peticiones Simples
Hacer una petición a la API es tan sencillo como llamar a un método. Vamos a empezar probando a pedirle algo simple.

In [ ]:
prompt_ejemplo = "¿En qué año se lanzó el primer iPhone de Apple?"
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt_ejemplo
)

print(f"Pregunta: {prompt_ejemplo}")
print("-" * 30)
print(response.text)

### Ejercicio 1: Prompting Estructurado
La estructura del prompt define completamente cómo hará la respuesta, un buen prompt mejora considerablemente la calidad.

**Tu turno:** Escribe un *prompt* que contenga varios elementos (como vimos en teoría: Rol, Tarea, Contexto, Formato). Pide, por ejemplo, que actúe como un experto en ciberseguridad, que te cuente en exactamente 3 frases cuál es el problema más común de las páginas web este año y que te devuelva el resultado envuelto en etiquetas HTML `<strong>`.

In [ ]:
#EJERCICIO 1: Completa el string con tu prompt y lánzaselo al modelo. 
mi_primer_prompt = """
(Escribe aquí tu prompt estructurado...)
"""

#Tu código aquí para lanzar la consulta

---
## 5. Parámetros del API: Temperature
Como comentamos en clase, el modelo no es 100% determinista. Podemos decirle mediante parámetros cómo queremos que sea de predecible o "creativo" usando el valor `temperature` (va desde 0.0 hasta 1.0+).  
Veamos cómo afecta a un mismo *prompt* lanzar la misma consulta dos veces con diferentes valores.

**Qué hay que hacer**:
  - Ejecuta el modelo con temperaturas: `0.0, 0.5, 1.0, 1.5, 2.0`
  - Observa cuándo la respuesta empieza a ser más extraña.
  - **Importante**: `config_temp` se pasa como `config=config_temp` dentro de `generate_content(...)`.

In [ ]:
from google.genai import types
#Vamos a pedirle un nombre creativo:
prompt_nombres = "Escribe 3 posibles nombres de fantasía para un gato mago."

#TODO: Prueba con varios niveles de temperatura (recomendable hacerlo con un bucle para no ejecutar varias veces la celda)
#Crea una configuración con GenerateContentConfig para añadir la temperatura



#TODO: lanza la consulta con el prompt
    
print(f"\n[ TEMPERATURE: {t} ] {'-'*30}")
print(response.text)

#Ejecuta esta celda varias veces (pero cuidado con el rate limit), observa cómo los de temp 0.0 casi siempre son los mismos o muy parecidos y los de temp 1.0 varían bastante más cada vez que la ejecutas.

---
## 6. Alucinaciones (mintiendo con muchísima confianza)
Si le pedimos algo para lo que no tiene información porque es un dato local tuyo, o es muy moderno y no está en sus datos de entrenamiento, se lo inventará (para completar probabilísticamente el texto).  
Esto es uno de los problemas fundamentales que siempre deben ser comprobados.

In [ ]:
#Hagamos una pregunta inventada:
prompt_alucinacion = "¿Cómo optimiza Cyberpunk 2077 la carga de NPCs usando streaming predictivo basado en comportamiento del jugador?"
response_alucinacion = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt_alucinacion,
)

print(response_alucinacion.text)

### Ejercicio 3: Mitigando la Alucinación
En la teoría vimos cómo **dar contexto** y **poner restricciones estrictas** puede evitar esto.

**Tu turno:** Modifica el prompt de arriba (o crea uno nuevo) sobre el mismo tema pero aplicando técnicas de mitigación para que el modelo te responda que no lo sabe o que esa información no existe, en lugar de inventársela.

In [ ]:
#EJERCICIO 3: Crea un prompt que mitigue la alucinación

prompt_mitigado = """

"""

#TODO: Tu código para llamar al modelo con este nuevo prompt

---
## 7. El LLM como herramienta "Todo en Uno"

En la práctica anterior con Hugging Face, necesitabas cargar **un pipeline y un modelo distinto** para cada tarea (analizar sentimiento, traducir, resumir, extraer entidades...). 

Con un LLM puedes hacer *todo* eso a la vez con un solo prompt.

### Ejercicio 4: Traducción y Resumen simultáneo
En un solo prompt, pásale el texto en inglés de abajo, pídele que extraiga sus **3 puntos clave**, y que te los devuelva directamente **traducidos al español**.

In [ ]:
#EJERCICIO 4: Traducción y Resumen en un solo paso

texto_ingles = """
The James Webb Space Telescope (JWST) is an infrared space observatory that launched on Dec 25, 2021. 
It is the largest and most powerful space telescope ever built. 
Its main goals are to study the first stars and galaxies that formed after the Big Bang, 
to observe planetary systems around other stars, and to explore the potential for life on exoplanets. 
Unlike the Hubble telescope, which orbits Earth, JWST orbits the Sun at the second Lagrange point (L2), 
keeping it permanently shaded from the Sun to maintain its ultra-cold operating temperature.
"""

prompt_todo_en_uno = f"""
(Escribe aquí tu prompt pidiendo: Extrae 3 puntos clave en español del texto provisto)

Texto:
{texto_ingles}
"""

#TODO: Tu código para llamar al modelo e imprimir el resultado:

---
### Ejercicio 5: Explicación y Generación de Código
Los modelos de lenguaje son excepcionales en tareas de programación. Una de sus aplicaciones estrella no es solo crear código de cero, sino **explicar, refactorizar y encontrar errores lógicos** o de seguridad (bugs) en código ya existente.

**Tu turno:** El siguiente fragmento de código está en un lenguaje esotérico llamado **Chef**, el cual es bastante confuso cuando lo leemos por primera vez. Antes de preguntar a Gemini, piensa, ¿qué crees que hace este código?

Después, diseña un prompt que evalúe y limpie este código, mejorándolo y traduciéndolo a un lenguaje que conozcas: pide a Gemini que te **explique brevemente qué intenta hacer** y después te devuelva el **código refactorizado en Python limpio**.

In [ ]:
#EJERCICIO 5: Refactorizar y explicar código sucio

codigo_sucio = """
Incident Response Risotto.

This recipe normalizes a tiny kitchen telemetry stream. 
Chef's note: plated values must be interpreted as ASCII.

Ingredients.
12 g harina
8 g huevo
9 g leche
17 g sal
31 g aceite
2 g aire
23 g wok
14 g romero
11 g laurel
4 g diente
1 g guindilla
99 g confeti
3 g offset
7 g humo

Method.
Put confeti into 2nd mixing bowl.
Add offset to confeti.
Subtract offset from confeti.
Clean 2nd mixing bowl.

Put harina into 1st mixing bowl.
Add 60 to harina.
Put huevo into 1st mixing bowl.
Add 93 to huevo.
Put leche into 1st mixing bowl.
Add 99 to leche.
Put sal into 1st mixing bowl.
Add 91 to sal.
Put aceite into 1st mixing bowl.
Add 80 to aceite.
Put aire into 1st mixing bowl.
Add 30 to aire.
Put wok into 1st mixing bowl.
Add 64 to wok.
Put aceite into 1st mixing bowl.
Add 80 to aceite.
Put romero into 1st mixing bowl.
Add 100 to romero.
Put laurel into 1st mixing bowl.
Add 97 to laurel.
Put diente into 1st mixing bowl.
Add 96 to diente.
Put guindilla into 1st mixing bowl.
Add 32 to guindilla.

Refrigerate for 0 hours.
Liquefy contents of the 1st mixing bowl.
Pour contents of the 1st mixing bowl into the baking dish.

Serve with parsley.
Serves 1.
"""

prompt_codigo = f"""
(Escribe aquí tu prompt pidiendo: explicar qué hace y presentar la versión en código limpio)

Código:
{codigo_sucio}
"""

#TODO: Tu código para llamar al modelo e imprimir el resultado

---
### Ejercicio 6: Análisis de Sentimiento y Clasificación Compleja

Anteriormente hicimos análisis de sentimiento en un pipeline y Extracción de Entidades Nombradas (NER) en otro diferente. Con un LLM puedes decirle en el mismo prompt que clasifique el sentimiento genérico y además extraiga datos precisos bajo el formato que tú elijas.

**Tu turno:** Diseña un prompt para que el modelo lea reviews de Steam, clasifique el sentimiento general (Positivo, Neutro o Negativo), e indique de forma separada el motivo de la queja / alabanza principal.

In [ ]:
#EJERCICIO 6: Clasificación y Extracción mixta

review_cliente_1 = "They turned me french during a mission"
review_cliente_2 = "I know Activision gets a lot of hate, but personally, I don't think it gets enough"
review_cliente_3 = "This community is so nice and helpful, I got a lot of tips on how to uninstall the game"


prompt_review = f"""
(Tu instrucción pidiendo: Sentimiento = NEGATIVO/POSITIVO/NEUTRO, Queja principal = 'Motivo...')

Review:
{review_cliente_1}

{review_cliente_2}

{review_cliente_3}
"""

#TODO: Tu código aquí

---
## 8. El modo Chat (Memoria en la Conversación)

Hasta ahora hemos usado peticiones *aisladas*: llamamos a `generate_content()`, la API recibe nuestra pregunta, emite una respuesta y **todo se olvida**. No hay memoria.

¿Y si queremos poder volver a preguntar, tal y cómo hacemos al escribirle a ChatGPT, donde se acuerda de lo que comentamos arriba? 
Para eso, emplearemos una función del SDK de Google que almacena todo lo que le escribimos como el contexto del chat.
Esto nos ahorra tener que incluir el contexto en cada prompt.

### Ejemplo: Chat añadiendo contexto
En este ejemplo se ve la forma típica de hacer un chat, indicando el contexto en cada prompt. Es útil en ciertos casos, pero Gemini ha añadido una nueva forma más sencilla.

In [ ]:
#Ejemplo incluyendo el contexto en cada prompt, sin usar chat

#Mensaje 1
contexto_1 = "El usuario dice: Me llamo Juanito y mi comida favorita son las croquetas."
pregunta_1 = "Responde amablemente a esta presentación en una frase."
prompt_1 = f"""
Contexto:
{contexto_1}

Pregunta:
{pregunta_1}
"""
resp_1 = client.models.generate_content(model=MODEL_ID, contents=prompt_1)
print("Respuesta 1:", resp_1.text)

#Mensaje 2 (hay que volver a pasar el contexto)
pregunta_2 = "¿Cómo me llamo y cuál es mi comida favorita?"
prompt_2 = f"""
Contexto:
{contexto_1}

Pregunta:
{pregunta_2}
"""
resp_2 = client.models.generate_content(model=MODEL_ID, contents=prompt_2)
print("Respuesta 2:", resp_2.text)

#Mensaje 3 (actualizamos contexto manualmente)
contexto_2 = contexto_1 + "\nEl modelo ha respondido: " + resp_1.text + "\nEl modelo ha respondido: " + resp_2.text
pregunta_3 = "¿Entonces qué quiere cenar hoy?"
prompt_3 = f"""
Contexto:
{contexto_2}

Pregunta:
{pregunta_3}
"""
resp_3 = client.models.generate_content(model=MODEL_ID, contents=prompt_3)
print("Respuesta 3:", resp_3.text)

### Ejercicio 7: Chateando con el modelo
Ahora se va a realizar esto mismo pero usando un chat, SIN necesidad de volver a enviarle el contexto. En la segunda consulta (en la que NO volvemos a repetir esa información), le pediremos al modelo que la recuerde. Para que este ejercicio funcione, tienes que enviar el mensaje usando un objeto especial `.send_message()` en vez de la función general del modelo base.

In [ ]:
#EJERCICIO 7: Iniciar una sesión mantenida de Chat (con Historial)

#TODO: Instancia una sesión nueva de chat, que arranca siempre con la memoria vacía


#TODO: prueba a mandar un mensaje y a continuar una conversación. Los mensajes siguientes al primero deberían requerir conocimientos de mensajes anteriores.


---
## 9. Function Calling

El Function Calling es lo que transforma a Gemini de un simple "lector de texto" a un Agente capaz de operar en el mundo real.
Gemini no sabe sumar números grandes perfectamente ni sabe qué hora es hoy, pero sabe cuándo pedirle a Python que lo haga por él.

¿Cómo funciona esto por dentro?
  - Análisis: El modelo lee la pregunta y detecta que necesita un dato que no tiene, pero ve que tiene una herramienta.
  - Llamada: Gemini genera un mensaje especial (no visible para el usuario) pidiendo los datos de esa función.
  - Ejecución: El SDK ejecuta el código de Python real.
  - Respuesta Final: Gemini recibe el resultado y redacta la respuesta final para el usuario

Esto es un "Agente de IA":
  - No alucina: No se inventa datos, lo saca de tu base de conocimientos.
  - Decide solo: Si le preguntas algo que no tiene que ver, no usará funciones porque sabe que no es relevante.
  - Encadena tareas: Gemini puede decidir llamar a dos funciones diferentes en orden para dar una sola respuesta.

## Ejercicio 8: Agente de IA
Se va a hacer un Agente de IA que permita gestionar una "Tienda de informática".

**Qué hay que hacer:**
 - Función obtener_precio_producto(nombre_producto) que devuelva el valor del producto (portatil/raton/teclado/monitor...)
 - Función calcular_descuento(precio, porcentaje) que devuelva el precio final después de un descuento
 - Añadir las herramientas a tools, en la configuración del chat (types.GenerateContentConfig)

In [ ]:
def obtener_precio_producto(producto: str) -> float:
    """Devuelve el precio de un producto tecnológico específico."""
    #TODO: Implementa esta función

def calcular_descuento(precio: float, porcentaje: float) -> float:
    """Calcula el precio final tras aplicar un descuento."""
    #TODO: Implementa esta función

#Metemos las funciones en una lista para dárselas a Gemini
mis_herramientas = [obtener_precio_producto, calcular_descuento]

Ahora le pasamos estas funciones a Gemini al crear el chat. El modelo no ejecuta la función, sino que responde con una instrucción diciendo: "Oye, necesito que ejecutes obtener_precio_producto para 'portatil'".

In [ ]:
#Iniciamos el chat con las herramientas activadas
chat = ...

#Probamos al agente
pregunta = "Hola, ¿cuánto cuesta el portatil y a cuánto se queda si me haces un 15% de descuento?"

#TODO: enviar la pregunta al agente y mostrar la respuesta